# Train nozzle_bidones_v11 (Colab)

Fine-tune **YOLOv8n** Bidon/Pico (`imgsz=416`).

**Dataset:** zip Roboflow YOLOv8 (`picos-bidones-v11.zip`). Preferir **Object Detection** con bbox ajustadas (sin mano dentro del Pico). Si los labels vienen como poligono/seg, la celda 2 los convierte a bbox detect.

- Roboflow: **sin Resize/Stretch** (originales).
- Incluir imagenes **negativas** (mano/sin pico) con `.txt` vacio.
- Clases: `0=Bidon`, `1=Pico` (Roboflow id 1 puede llamarse `nozzle`).
- Runtime: Runtime → Change runtime type → **GPU**.

Salidas: `nozzle_bidones_v11` / `yolov8n_nozzle_bidones_v11.onnx` → calib/RKNN en WSL.

## 1) Setup

In [ ]:
# GPU recomendada: Runtime -> Change runtime type -> GPU
!pip -q install -U ultralytics opencv-python-headless
import torch
from ultralytics import YOLO
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2) Subir, descomprimir y convertir labels (seg → detect si hace falta)

Subi el zip Roboflow, ej. `picos-bidones-v11.zip` (`train/images`, `train/labels`, `valid/...`).

Si los `.txt` tienen poligonos, se genera `/content/picos-bidones-v11_detect` con bbox AABB antes de entrenar.

In [ ]:
from pathlib import Path
import shutil
import zipfile

DATASET_VER = "v11"
SPLITS = ("train", "valid", "test")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _polygon_to_xywh(coords: list[float]):
    if len(coords) < 4 or len(coords) % 2 != 0:
        return None
    xs = coords[0::2]
    ys = coords[1::2]
    x_min = max(0.0, min(xs))
    x_max = min(1.0, max(xs))
    y_min = max(0.0, min(ys))
    y_max = min(1.0, max(ys))
    w = x_max - x_min
    h = y_max - y_min
    if w <= 0.0 or h <= 0.0:
        return None
    return x_min + w / 2.0, y_min + h / 2.0, w, h


def _line_to_detect(line: str) -> str | None:
    parts = line.strip().split()
    if not parts:
        return None
    try:
        cls_id = int(float(parts[0]))
    except ValueError:
        return None
    nums = [float(x) for x in parts[1:]]
    if len(nums) == 4:
        cx, cy, w, h = nums
    else:
        box = _polygon_to_xywh(nums)
        if box is None:
            return None
        cx, cy, w, h = box
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def _label_needs_convert(label_dir: Path) -> bool:
    for lbl in label_dir.glob("*.txt"):
        for line in lbl.read_text(encoding="utf-8").splitlines():
            if line.strip() and len(line.split()) != 5:
                return True
    return False


def _convert_dataset(src_root: Path, dst_root: Path) -> tuple[int, int, int]:
    n_images = n_labels = n_boxes = 0
    for split in SPLITS:
        src_img = src_root / split / "images"
        src_lbl = src_root / split / "labels"
        dst_img = dst_root / split / "images"
        dst_lbl = dst_root / split / "labels"
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        if not src_img.is_dir():
            continue
        for img_path in sorted(src_img.iterdir()):
            if not img_path.is_file() or img_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            shutil.copy2(img_path, dst_img / img_path.name)
            n_images += 1
            lbl_src = src_lbl / f"{img_path.stem}.txt"
            lbl_dst = dst_lbl / f"{img_path.stem}.txt"
            out_lines: list[str] = []
            if lbl_src.is_file():
                for raw in lbl_src.read_text(encoding="utf-8").splitlines():
                    converted = _line_to_detect(raw)
                    if converted is not None:
                        out_lines.append(converted)
                        n_boxes += 1
            lbl_dst.write_text(
                ("\n".join(out_lines) + "\n") if out_lines else "",
                encoding="utf-8",
            )
            n_labels += 1
    return n_images, n_labels, n_boxes


# Colab: zip ya en /content/ o subir desde PC
ZIP_CANDIDATES = (
    f"/content/picos-bidones-{DATASET_VER}.zip",
    f"/content/picos-bidones-{DATASET_VER}_detect.zip",
)
zip_path = next((p for p in ZIP_CANDIDATES if Path(p).is_file()), None)
if zip_path is None:
    from google.colab import files

    uploaded = files.upload()
    zip_path = "/content/" + next(iter(uploaded.keys()))

print("ZIP =", zip_path)

RAW_ROOT = Path(f"/content/picos-bidones-{DATASET_VER}_raw")
if RAW_ROOT.exists():
    shutil.rmtree(RAW_ROOT)
RAW_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(RAW_ROOT)

kids = [p for p in RAW_ROOT.iterdir() if p.is_dir()]
if len(kids) == 1 and (kids[0] / "train").is_dir():
    RAW_ROOT = kids[0]

print("RAW_ROOT =", RAW_ROOT)
print("train images:", len(list((RAW_ROOT / "train" / "images").glob("*"))))
print("valid images:", len(list((RAW_ROOT / "valid" / "images").glob("*"))))

needs_convert = _label_needs_convert(RAW_ROOT / "train" / "labels")
DATA_ROOT = Path(f"/content/picos-bidones-{DATASET_VER}_detect")

if needs_convert:
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    n_img, n_lbl, n_box = _convert_dataset(RAW_ROOT, DATA_ROOT)
    print("Labels poligono/seg -> bbox detect")
    print(f"  imagenes={n_img}  label files={n_lbl}  boxes={n_box}")
else:
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    shutil.copytree(RAW_ROOT, DATA_ROOT)
    print("Labels ya son bbox YOLO-detect (sin conversion)")

print("DATA_ROOT (entrenamiento) =", DATA_ROOT)

sample = next((DATA_ROOT / "train" / "labels").glob("*.txt"), None)
if sample is not None:
    line = sample.read_text(encoding="utf-8").strip().splitlines()[0]
    n = len(line.split())
    print(
        "sample label tokens:",
        n,
        "|->",
        "OK bbox YOLO-detect" if n == 5 else "ERROR: revisar conversion",
    )

## 3) data.yaml (nombres de producto)

In [ ]:
from pathlib import Path

DATA_YAML = Path("/content/data_nozzle_bidones_v11.yaml")
DATA_YAML.write_text(
    f"""path: {DATA_ROOT.resolve().as_posix()}
train: train/images
val: valid/images
test: test/images

nc: 2
names:
  - Bidon
  - Pico
""",
    encoding="utf-8",
)
print(DATA_YAML.read_text(encoding="utf-8"))

## 4) Entrenar YOLOv8n @416

Letterbox lo hace Ultralytics (no hace falta resize en Roboflow).

In [ ]:
from pathlib import Path
from ultralytics import YOLO

IMGSZ = 416
EPOCHS = 100
BATCH = 16
PATIENCE = 20
CLOSE_MOSAIC = 10
RUN_NAME = "nozzle_bidones_v11"

model = YOLO("yolov8n.pt")
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    close_mosaic=CLOSE_MOSAIC,
    project="/content/runs/detect",
    name=RUN_NAME,
    exist_ok=True,
)

BEST_PT = Path("/content/runs/detect") / RUN_NAME / "weights" / "best.pt"
print("best.pt =", BEST_PT, "| exists:", BEST_PT.is_file())

## 5) Export ONNX + descargar artefactos

En el repo local: copiar `best.pt` a `yolo_train/runs/detect/nozzle_bidones_v11/weights/` y el ONNX a `Yolo-Weights/yolov8n_nozzle_bidones_v11.onnx`, luego calib/RKNN en WSL.

In [ ]:
from google.colab import files
from pathlib import Path
from ultralytics import YOLO
import shutil

RUN_NAME = "nozzle_bidones_v11"
BEST_PT = Path("/content/runs/detect") / RUN_NAME / "weights" / "best.pt"
assert BEST_PT.is_file(), f"No esta {BEST_PT}; corre la celda de train"

IMGSZ = 416
ONNX_OUT = Path("/content/yolov8n_nozzle_bidones_v11.onnx")

model = YOLO(str(BEST_PT))
exported = Path(model.export(format="onnx", imgsz=IMGSZ, opset=19))
if exported.resolve() != ONNX_OUT.resolve():
    shutil.copy2(exported, ONNX_OUT)

print("OK pt  ->", BEST_PT)
print("OK onnx->", ONNX_OUT)

files.download(str(BEST_PT))
files.download(str(ONNX_OUT))

## Checklist post-Colab

1. Actualizar `yolo_train/nozzle_config.py` a **v11** (dataset + nombres de artefactos).
2. WSL `venv-rknn310`: `prepare_nozzle_calib_v8.py` (calib apunta a v11 en config).
3. WSL `venv-rknn311`: `exp_yolov8n_nozzle_rknn_v8.py`.
4. Placa: `NOZZLE_MODEL_RK3568=models/yolov8n_nozzle_bidones_v11.rknn`.